In [ ]:
!pip install -q --upgrade pip setuptools wheel
!pip install -q seqeval transformers accelerate torch sentencepiece bitsandbytes

In [ ]:
"""
Few-shot benchmarking script for md-nishat-008/TigerLLM-9B-it (8-bit quantized)
on a Bangla Biomedical NER dataset (IOB2 token-level tagging).

Few-shot strategy:
- Automatically selects one short demonstration example for each entity class.
- Also selects one short O-only example when available.
- Demonstration rows are excluded from evaluation to avoid data leakage.
- The same fixed few-shot examples are used for every evaluation sentence.
- Optimized for 2xT4 GPU with batched inference.
"""

import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import re
import time
import traceback
import warnings
import logging
import pandas as pd
import torch

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)


# --------------------------------------------------------------------------- #
# Configuration
# --------------------------------------------------------------------------- #

MODEL_NAME = "md-nishat-008/TigerLLM-9B-it"

DATASET_PATH = "/kaggle/input/datasets/syedmdnafissameen/ner-200/NER_stratified_sample_200.csv"
LOCAL_DATASET_FALLBACK = "NER_stratified_sample_200.csv"

OUTPUT_PREDICTIONS_CSV = "fewshot_predictions.csv"
OUTPUT_RESULTS_CSV = "fewshot_benchmark_results.csv"
OUTPUT_FEWSHOT_CSV = "fewshot_examples.csv"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# None = evaluate all rows except rows used as few-shot demonstrations
NUM_SAMPLES = None
RANDOM_SAMPLE = False
SAMPLE_SEED = 42

BATCH_SIZE = 4
CHECKPOINT_EVERY_N_BATCHES = 25

MAX_NEW_TOKENS_CAP = 200

# One demonstration for every available primary entity class.
# Dataset contains:
# Symptom, Medicine, Specialist, Health Condition,
# Age, Medical Procedure, Dosage
# 3 compact demonstrations that collectively cover all 7 entity types
FEW_SHOT_INDICES = [24, 107, 134]

VALID_ENTITY_TYPES = {
    "Symptom",
    "Health_Condition",
    "Medicine",
    "Dosage",
    "Medical_Procedure",
    "Specialist",
    "Age",
}

VALID_TAG_PATTERN = re.compile(
    r"^(O|[BI]-(Symptom|Health_Condition|Medicine|Dosage|"
    r"Medical_Procedure|Specialist|Age))$"
)

_LABEL_START_RE = re.compile(r"^(O|[BI]-.+)$")


# --------------------------------------------------------------------------- #
# System prompt
# --------------------------------------------------------------------------- #

SYSTEM_PROMPT = """You are a Bangla Biomedical Named Entity Recognition (BioNER) model.

Your task is to assign exactly one IOB2 label to every input token.

Allowed entity types and labels:

Symptom:
B-Symptom
I-Symptom

Health Condition:
B-Health_Condition
I-Health_Condition

Medicine:
B-Medicine
I-Medicine

Dosage:
B-Dosage
I-Dosage

Medical Procedure:
B-Medical_Procedure
I-Medical_Procedure

Specialist:
B-Specialist
I-Specialist

Age:
B-Age
I-Age

Outside any biomedical entity:
O

Strict rules:

1. Output exactly one label for each input token.
2. Preserve the original token order.
3. Use B- for the first token of an entity span.
4. Use I- for continuation tokens of the same entity span.
5. Use O for every token that is not part of an entity.
6. Do not merge tokens.
7. Do not skip tokens.
8. Do not add tokens.
9. Return ONLY the labels.
10. Put one label on each line.
11. Do not include token numbers.
12. Do not include explanations.
13. Do not include markdown.
14. Do not output any text before or after the labels.

Study the provided demonstrations carefully and follow the same labeling format."""


# --------------------------------------------------------------------------- #
# Dataset utilities
# --------------------------------------------------------------------------- #

def load_dataset(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        if os.path.exists(LOCAL_DATASET_FALLBACK):
            path = LOCAL_DATASET_FALLBACK
        else:
            raise FileNotFoundError(
                f"Dataset not found at:\n"
                f"{path}\n"
                f"or fallback:\n"
                f"{LOCAL_DATASET_FALLBACK}"
            )

    df = pd.read_csv(path)

    required_cols = {"text", "labels", "primary_class"}
    missing = required_cols - set(df.columns)

    if missing:
        raise ValueError(
            f"Dataset missing required columns: {missing}"
        )

    df = df.copy()
    df["original_index"] = df.index

    return df


def tokenize_text(text):
    return str(text).split()


def fix_true_labels(label_str):
    if pd.isna(label_str):
        return []

    raw = str(label_str).split()
    fixed = []

    for tok in raw:
        if _LABEL_START_RE.match(tok):
            fixed.append(tok)
        else:
            if fixed:
                fixed[-1] = fixed[-1] + "_" + tok
            else:
                fixed.append(tok)

    return fixed


def align_labels(tokens, labels):
    labels = list(labels)

    if len(labels) < len(tokens):
        labels.extend(
            ["O"] * (len(tokens) - len(labels))
        )

    elif len(labels) > len(tokens):
        labels = labels[:len(tokens)]

    return labels


def prepare_row(row):
    tokens = tokenize_text(row["text"])
    labels = fix_true_labels(row["labels"])
    labels = align_labels(tokens, labels)

    return tokens, labels


# --------------------------------------------------------------------------- #
# Automatic few-shot example selection
# --------------------------------------------------------------------------- #

def select_few_shot_examples(df):

    examples = []
    selected_indices = set()

    for original_index in FEW_SHOT_INDICES:

        matching = df[
            df["original_index"] == original_index
        ]

        if len(matching) == 0:
            raise ValueError(
                f"Few-shot row {original_index} not found."
            )

        row = matching.iloc[0]

        tokens, labels = prepare_row(row)

        examples.append(
            {
                "original_index": original_index,
                "primary_class": row["primary_class"],
                "tokens": tokens,
                "labels": labels,
                "text": row["text"],
            }
        )

        selected_indices.add(original_index)

    demo_records = []

    for i, example in enumerate(examples, start=1):

        demo_records.append(
            {
                "shot_number": i,
                "original_index": example["original_index"],
                "primary_class": example["primary_class"],
                "text": example["text"],
                "labels": " ".join(example["labels"]),
                "token_count": len(example["tokens"]),
            }
        )

    pd.DataFrame(demo_records).to_csv(
        OUTPUT_FEWSHOT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== Few-Shot Demonstrations =====")

    for i, example in enumerate(examples, start=1):
        print(
            f"Shot {i}: "
            f"class={example['primary_class']}, "
            f"tokens={len(example['tokens'])}, "
            f"dataset_index={example['original_index']}"
        )

    print(
        f"\n[INFO] Total few-shot demonstrations: "
        f"{len(examples)}"
    )

    return examples, selected_indices

# --------------------------------------------------------------------------- #
# Prompt formatting
# --------------------------------------------------------------------------- #

def format_token_message(tokens):
    numbered_tokens = "\n".join(
        f"{i + 1}. {token}"
        for i, token in enumerate(tokens)
    )

    return (
        f"TOKENS:\n"
        f"{numbered_tokens}\n\n"
        f"OUTPUT "
        f"(exactly {len(tokens)} labels, "
        f"one per line, nothing else):"
    )


def format_label_answer(labels):
    return "\n".join(labels)


def build_prompt(tokens):
    return format_token_message(tokens)


def build_few_shot_messages(
    few_shot_examples,
    target_user_message
):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        }
    ]

    for example in few_shot_examples:

        messages.append(
            {
                "role": "user",
                "content": format_token_message(
                    example["tokens"]
                ),
            }
        )

        messages.append(
            {
                "role": "assistant",
                "content": format_label_answer(
                    example["labels"]
                ),
            }
        )

    messages.append(
        {
            "role": "user",
            "content": target_user_message,
        }
    )

    return messages


def build_fallback_prompt(
    few_shot_examples,
    target_user_message
):
    parts = [SYSTEM_PROMPT]

    for i, example in enumerate(
        few_shot_examples,
        start=1
    ):

        parts.append(
            f"\n\nDEMONSTRATION {i}\n\n"
            f"{format_token_message(example['tokens'])}\n"
            f"{format_label_answer(example['labels'])}"
        )

    parts.append(
        "\n\nNOW LABEL THE FOLLOWING INPUT.\n\n"
        + target_user_message
    )

    return "\n".join(parts)


def _build_chat_prompt(
    tokenizer,
    user_message,
    few_shot_examples
):
    messages = build_few_shot_messages(
        few_shot_examples,
        user_message
    )

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

    except Exception:

        fallback_content = build_fallback_prompt(
            few_shot_examples,
            user_message
        )

        try:
            fallback_messages = [
                {
                    "role": "user",
                    "content": fallback_content,
                }
            ]

            return tokenizer.apply_chat_template(
                fallback_messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        except Exception:
            return fallback_content


# --------------------------------------------------------------------------- #
# Model loading
# --------------------------------------------------------------------------- #

def load_model_and_tokenizer(model_name):
    print(
        f"[INFO] Loading tokenizer for "
        f"{model_name} ..."
    )

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    print(
        f"[INFO] Loading model {model_name} "
        f"in 8-bit on device={DEVICE} ..."
    )

    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=(
            quant_config
            if DEVICE == "cuda"
            else None
        ),
        device_map=(
            "auto"
            if DEVICE == "cuda"
            else None
        ),
        trust_remote_code=True,
    )

    model.eval()

    print("[INFO] Model loaded successfully.")

    return tokenizer, model


# --------------------------------------------------------------------------- #
# Batched inference
# --------------------------------------------------------------------------- #

def run_inference_batch(
    model,
    tokenizer,
    user_messages,
    few_shot_examples,
    max_new_tokens,
):

    prompts = [
        _build_chat_prompt(
            tokenizer,
            user_message,
            few_shot_examples
        )
        for user_message in user_messages
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=4096,
        add_special_tokens=False,
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            temperature=None,
            top_p=None,
            top_k=None,
            pad_token_id=(
                tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id
            ),
            eos_token_id=tokenizer.eos_token_id,
        )

    input_len = inputs["input_ids"].shape[1]

    decoded = []

    for i in range(len(prompts)):

        gen_tokens = output_ids[i][input_len:]

        text = tokenizer.decode(
            gen_tokens,
            skip_special_tokens=True
        ).strip()

        decoded.append(text)

    return decoded


# --------------------------------------------------------------------------- #
# Prediction parsing
# --------------------------------------------------------------------------- #

def normalize_label(label):
    label = str(label).strip()

    label = re.sub(
        r"^\d+[\.\)]\s*",
        "",
        label
    )

    label = (
        label
        .strip()
        .strip('"')
        .strip("'")
        .strip("`")
    )

    if not label:
        return "O"

    # Normalize hyphen variants.
    label = (
        label
        .replace("–", "-")
        .replace("—", "-")
    )

    if label.upper() == "O":
        return "O"

    match = re.match(
        r"^([BbIi])[-_ ](.+)$",
        label
    )

    if not match:
        return "O"

    prefix = match.group(1).upper()

    entity = match.group(2).strip()

    entity = entity.replace(" ", "_")
    entity = entity.replace("-", "_")

    normalized_entity_map = {
        "symptom": "Symptom",
        "health_condition": "Health_Condition",
        "healthcondition": "Health_Condition",
        "medicine": "Medicine",
        "dosage": "Dosage",
        "medical_procedure": "Medical_Procedure",
        "medicalprocedure": "Medical_Procedure",
        "specialist": "Specialist",
        "age": "Age",
    }

    entity_key = entity.lower()

    if entity_key not in normalized_entity_map:
        return "O"

    entity = normalized_entity_map[entity_key]

    candidate = f"{prefix}-{entity}"

    if VALID_TAG_PATTERN.match(candidate):
        return candidate

    return "O"


def extract_possible_label(line):
    line = line.strip()

    if not line:
        return None

    # Remove common numbering/bullet prefixes.
    line = re.sub(
        r"^\s*[\-\*\•]*\s*",
        "",
        line
    )

    line = re.sub(
        r"^\d+[\.\):\-]\s*",
        "",
        line
    )

    # Direct match first.
    direct = normalize_label(line)

    if direct != "O":
        return direct

    if line.strip().upper() == "O":
        return "O"

    # Search for a label inside accidental extra text.
    pattern = re.compile(
        r"\b("
        r"O|"
        r"[BI][-_ ]"
        r"(?:"
        r"Symptom|"
        r"Health[_ ]Condition|"
        r"Medicine|"
        r"Dosage|"
        r"Medical[_ ]Procedure|"
        r"Specialist|"
        r"Age"
        r")"
        r")\b",
        flags=re.IGNORECASE,
    )

    match = pattern.search(line)

    if match:
        return normalize_label(match.group(1))

    return None


def parse_predictions(
    raw_output,
    num_tokens
):

    lines = [
        line.strip()
        for line in str(raw_output).splitlines()
        if line.strip()
    ]

    labels = []

    for line in lines:

        candidate = extract_possible_label(line)

        if candidate is not None:
            labels.append(candidate)

    if len(labels) < num_tokens:
        labels.extend(
            ["O"] * (num_tokens - len(labels))
        )

    elif len(labels) > num_tokens:
        labels = labels[:num_tokens]

    return labels


# --------------------------------------------------------------------------- #
# Benchmarking
# --------------------------------------------------------------------------- #

def benchmark(
    model,
    tokenizer,
    df,
    few_shot_examples,
):

    all_true_labels = []
    all_pred_labels = []

    latencies = []
    records = []

    total_correct_tokens = 0
    total_tokens = 0

    num_batches = (
        len(df) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    print(
        f"\n[INFO] Evaluation sentences: "
        f"{len(df)}"
    )

    print(
        f"[INFO] Few-shot demonstrations per prompt: "
        f"{len(few_shot_examples)}"
    )

    for batch_idx, start in enumerate(
        range(0, len(df), BATCH_SIZE)
    ):

        batch_df = df.iloc[
            start:start + BATCH_SIZE
        ]

        batch_texts = (
            batch_df["text"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        batch_primary = (
            batch_df["primary_class"]
            .fillna("")
            .astype(str)
            .tolist()
        )

        batch_original_indices = (
            batch_df["original_index"]
            .tolist()
        )

        batch_tokens = [
            tokenize_text(text)
            for text in batch_texts
        ]

        batch_true_labels = []

        for _, row in batch_df.iterrows():

            tokens, labels = prepare_row(row)

            batch_true_labels.append(labels)

        prompts = [
            build_prompt(tokens)
            for tokens in batch_tokens
        ]

        longest_sentence = max(
            len(tokens)
            for tokens in batch_tokens
        )

        # A tag such as B-Medical_Procedure may require
        # multiple tokenizer tokens, so allocate more than
        # one generation token per input token.
        max_new = min(
            MAX_NEW_TOKENS_CAP,
            max(
                50,
                longest_sentence * 4 + 20
            )
        )

        start_t = time.time()

        try:

            raw_outputs = run_inference_batch(
                model=model,
                tokenizer=tokenizer,
                user_messages=prompts,
                few_shot_examples=few_shot_examples,
                max_new_tokens=max_new,
            )

            batch_error = ""

        except Exception as e:

            traceback.print_exc()

            batch_error = (
                f"{type(e).__name__}: {e}"
            )

            raw_outputs = [
                ""
                for _ in prompts
            ]

        elapsed = time.time() - start_t

        latency_per_sample = (
            elapsed / len(prompts)
            if len(prompts) > 0
            else 0.0
        )

        for i, tokens in enumerate(batch_tokens):

            pred_labels = parse_predictions(
                raw_outputs[i],
                len(tokens)
            )

            true_labels = batch_true_labels[i]

            correct = sum(
                1
                for true_label, pred_label
                in zip(
                    true_labels,
                    pred_labels
                )
                if true_label == pred_label
            )

            total_correct_tokens += correct
            total_tokens += len(tokens)

            all_true_labels.append(
                true_labels
            )

            all_pred_labels.append(
                pred_labels
            )

            latencies.append(
                latency_per_sample
            )

            records.append(
                {
                    "original_index":
                        batch_original_indices[i],

                    "text":
                        batch_texts[i],

                    "true_labels":
                        " ".join(true_labels),

                    "pred_labels":
                        " ".join(pred_labels),

                    "primary_class":
                        batch_primary[i],

                    "token_count":
                        len(tokens),

                    "correct_tokens":
                        correct,

                    "token_accuracy":
                        (
                            correct / len(tokens)
                            if len(tokens) > 0
                            else 0.0
                        ),

                    "latency":
                        latency_per_sample,

                    "raw_output":
                        raw_outputs[i],

                    "error":
                        batch_error,
                }
            )

        processed = min(
            start + BATCH_SIZE,
            len(df)
        )

        print(
            f"[INFO] Processed "
            f"{processed}/{len(df)} sentences "
            f"(batch "
            f"{batch_idx + 1}/{num_batches}, "
            f"{elapsed:.2f}s)"
        )

        if (
            batch_idx + 1
        ) % CHECKPOINT_EVERY_N_BATCHES == 0:

            pd.DataFrame(records).to_csv(
                OUTPUT_PREDICTIONS_CSV,
                index=False,
                encoding="utf-8-sig",
            )

            print(
                f"[INFO] Checkpoint saved "
                f"at batch {batch_idx + 1}"
            )

    # ------------------------------------------------------------------- #
    # Save predictions
    # ------------------------------------------------------------------- #

    predictions_df = pd.DataFrame(
        records
    )

    predictions_df.to_csv(
        OUTPUT_PREDICTIONS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[INFO] Saved predictions to "
        f"{OUTPUT_PREDICTIONS_CSV}"
    )

    # ------------------------------------------------------------------- #
    # Metrics
    # ------------------------------------------------------------------- #

    token_accuracy = (
        total_correct_tokens / total_tokens
        if total_tokens > 0
        else 0.0
    )

    precision_weighted = precision_score(
        all_true_labels,
        all_pred_labels,
        average="weighted",
        zero_division=0,
    )

    recall_weighted = recall_score(
        all_true_labels,
        all_pred_labels,
        average="weighted",
        zero_division=0,
    )

    f1_weighted = f1_score(
        all_true_labels,
        all_pred_labels,
        average="weighted",
        zero_division=0,
    )

    precision_micro = precision_score(
        all_true_labels,
        all_pred_labels,
        average="micro",
        zero_division=0,
    )

    recall_micro = recall_score(
        all_true_labels,
        all_pred_labels,
        average="micro",
        zero_division=0,
    )

    f1_micro = f1_score(
        all_true_labels,
        all_pred_labels,
        average="micro",
        zero_division=0,
    )

    precision_macro = precision_score(
        all_true_labels,
        all_pred_labels,
        average="macro",
        zero_division=0,
    )

    recall_macro = recall_score(
        all_true_labels,
        all_pred_labels,
        average="macro",
        zero_division=0,
    )

    f1_macro = f1_score(
        all_true_labels,
        all_pred_labels,
        average="macro",
        zero_division=0,
    )

    avg_latency = (
        sum(latencies) / len(latencies)
        if latencies
        else 0.0
    )

    report = classification_report(
        all_true_labels,
        all_pred_labels,
        zero_division=0,
    )

    print(
        "\n"
        "===== FEW-SHOT BENCHMARK METRICS ====="
    )

    print(
        f"Token Accuracy       : "
        f"{token_accuracy:.4f}"
    )

    print(
        f"Weighted Precision   : "
        f"{precision_weighted:.4f}"
    )

    print(
        f"Weighted Recall      : "
        f"{recall_weighted:.4f}"
    )

    print(
        f"Weighted F1          : "
        f"{f1_weighted:.4f}"
    )

    print(
        f"Micro Precision      : "
        f"{precision_micro:.4f}"
    )

    print(
        f"Micro Recall         : "
        f"{recall_micro:.4f}"
    )

    print(
        f"Micro F1             : "
        f"{f1_micro:.4f}"
    )

    print(
        f"Macro Precision      : "
        f"{precision_macro:.4f}"
    )

    print(
        f"Macro Recall         : "
        f"{recall_macro:.4f}"
    )

    print(
        f"Macro F1             : "
        f"{f1_macro:.4f}"
    )

    print(
        f"Avg Inference Latency: "
        f"{avg_latency:.4f} s"
    )

    print(
        "\n"
        "===== seqeval Classification Report ====="
    )

    print(report)

    # ------------------------------------------------------------------- #
    # Results CSV
    # ------------------------------------------------------------------- #

    results_df = pd.DataFrame(
        [
            {
                "model":
                    MODEL_NAME,

                "prompting":
                    "few-shot",

                "num_few_shot_examples":
                    len(few_shot_examples),

                "token_accuracy":
                    token_accuracy,

                "precision_weighted":
                    precision_weighted,

                "recall_weighted":
                    recall_weighted,

                "f1_weighted":
                    f1_weighted,

                "precision_micro":
                    precision_micro,

                "recall_micro":
                    recall_micro,

                "f1_micro":
                    f1_micro,

                "precision_macro":
                    precision_macro,

                "recall_macro":
                    recall_macro,

                "f1_macro":
                    f1_macro,

                "total_sentences":
                    len(df),

                "total_tokens":
                    total_tokens,

                "avg_latency_s":
                    avg_latency,
            }
        ]
    )

    results_df.to_csv(
        OUTPUT_RESULTS_CSV,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"[INFO] Saved benchmark results to "
        f"{OUTPUT_RESULTS_CSV}"
    )

    return predictions_df, results_df


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():

    print(
        f"[INFO] Using device: "
        f"{DEVICE}"
    )

    print(
        f"[INFO] GPUs available: "
        f"{torch.cuda.device_count()}"
    )

    # ------------------------------------------------------------------- #
    # Load full dataset
    # ------------------------------------------------------------------- #

    full_df = load_dataset(
        DATASET_PATH
    )

    print(
        f"[INFO] Loaded dataset with "
        f"{len(full_df)} rows."
    )

    # ------------------------------------------------------------------- #
    # Select fixed few-shot demonstrations
    # ------------------------------------------------------------------- #

    few_shot_examples, demo_indices = (
        select_few_shot_examples(
            full_df
        )
    )

    # ------------------------------------------------------------------- #
    # IMPORTANT:
    # Remove few-shot demonstration rows from evaluation.
    # This prevents the model from being tested on examples it saw
    # directly inside the prompt.
    # ------------------------------------------------------------------- #

    eval_df = full_df[
        ~full_df["original_index"].isin(
            demo_indices
        )
    ].copy()

    eval_df = eval_df.reset_index(
        drop=True
    )

    print(
        f"[INFO] Removed "
        f"{len(demo_indices)} demonstration rows "
        f"from evaluation."
    )

    print(
        f"[INFO] Remaining evaluation rows: "
        f"{len(eval_df)}"
    )

    # ------------------------------------------------------------------- #
    # Sort evaluation samples by token length for efficient batching
    # ------------------------------------------------------------------- #

    eval_df["_token_count"] = (
    eval_df["text"]
        .fillna("")
        .astype(str)
        .apply(lambda x: len(x.split()))
    )

    eval_df = (
        eval_df
        .sort_values("_token_count")
        .drop(columns=["_token_count"])
        .reset_index(drop=True)
    )

    print("[INFO] Evaluation rows sorted by token length.")

    # ------------------------------------------------------------------- #
    # Optional evaluation subset
    # ------------------------------------------------------------------- #

    if (
        NUM_SAMPLES is not None
        and NUM_SAMPLES < len(eval_df)
    ):

        if RANDOM_SAMPLE:

            eval_df = eval_df.sample(
                n=NUM_SAMPLES,
                random_state=SAMPLE_SEED,
            ).reset_index(drop=True)

            print(
                f"[INFO] Randomly sampled "
                f"{NUM_SAMPLES} evaluation rows "
                f"(seed={SAMPLE_SEED})."
            )

        else:

            eval_df = eval_df.iloc[
                :NUM_SAMPLES
            ].reset_index(drop=True)

            print(
                f"[INFO] Using first "
                f"{NUM_SAMPLES} evaluation rows."
            )

    # ------------------------------------------------------------------- #
    # Load model
    # ------------------------------------------------------------------- #

    tokenizer, model = (
        load_model_and_tokenizer(
            MODEL_NAME
        )
    )

    # ------------------------------------------------------------------- #
    # Run few-shot benchmark
    # ------------------------------------------------------------------- #

    benchmark(
        model=model,
        tokenizer=tokenizer,
        df=eval_df,
        few_shot_examples=few_shot_examples,
    )


if __name__ == "__main__":
    main()